In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter

# =====================================================================
# RPY2 & R INTERFACE SETUP 
# =====================================================================
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

survival_r = importr("survival")
survminer_r = importr("survminer")

warnings.filterwarnings('ignore')

from Library.ODE_timeseries import get_diagnosis_state, run_phased_simulation, generate_patient_fit_plots

# =====================================================================
# 1. BASELINE CONFIGURATION & OPTIMAL CUTPOINT
# =====================================================================
base_params = {
    'p1': 0.0152, 'd4': 2.3, 'd1': 0.0, 'd4l': 2.3,
    'k': 7.5e-9, 'tilde_n': 10.0, 'niche_capacity': 2.42105e11,
    'gamma': 7.5e-07,
    'p1l': 0.0, 'p_LSCbyHSC': 0.05, 'p_HSCbyLSC': 0.95,
    'healthy_neut_per_kg': 4e8
}

def extract_schedule(patient_id, cycles_df):
    id_col = 'Pseudonym' if 'Pseudonym' in cycles_df.columns else ('Patient_ID' if 'Patient_ID' in cycles_df.columns else 'ID')
    drug_data = cycles_df[cycles_df[id_col].astype(str).str.strip() == str(patient_id).strip()].copy()
    schedule = []
    for _, row in drug_data.iterrows():
        starts = [float(x) for x in [row.get('venetoclax_start'), row.get('aza_start')] if pd.notna(x)]
        stops  = [float(x) for x in [row.get('venetoclax_stop'), row.get('aza_stop')] if pd.notna(x)]
        if starts and stops:
            start, stop = min(starts), max(stops)
            if start <= stop and stop >= 0:
                schedule.append((start, stop))
    return sorted(schedule, key=lambda x: x[0])

def find_optimal_cutpoint(df, score_col, time_col, event_col, minprop=0.1):
    """Directly invokes R's survminer::surv_cutpoint() via rpy2 using localconverter."""
    sub_df = df[[time_col, event_col, score_col]].dropna().copy()
    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(sub_df)
    ro.globalenv['r_df'] = r_df

    r_code = f"""
    res_cut <- survminer::surv_cutpoint(
        data = r_df,
        time = "{time_col}",
        event = "{event_col}",
        variables = "{score_col}",
        minprop = {minprop}
    )
    cut_val <- res_cut$cutpoint$cutpoint
    r_df$grp <- ifelse(r_df${score_col} <= cut_val, 0, 1)
    diff <- survival::survdiff(survival::Surv({time_col}, {event_col}) ~ grp, data = r_df)
    pval <- 1 - pchisq(diff$chisq, df = length(diff$n) - 1)
    c(cut_val, pval)
    """
    r_res = ro.r(r_code)
    return float(r_res[0]), float(r_res[1])

# =====================================================================
# 2. GENERATE 3-PANEL FIGURE
# =====================================================================
def generate_figure_three_panels(merged_df, target_patient_id=9851,
                                 excel_path="InputData/treatment_trajectories.xlsx",
                                 output_path="Figures/Figure4.png"):
    sns.set_theme(style="whitegrid", rc={"grid.color": "#e6e6e6", "axes.edgecolor": "#cccccc", "axes.linewidth": 1.3})
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['pdf.fonttype'] = 42
    c_blue, c_orange, c_drug = "#0072B2", "#D55E00", "#dedede"

    # --- PANEL A: TRAJECTORY ---
    blood_raw = pd.read_excel(excel_path, sheet_name='Neutrophil')
    bm_raw = pd.read_excel(excel_path, sheet_name='BM_blast')
    drug_raw = pd.read_excel(excel_path, sheet_name='Cycles')

    p_id = str(target_patient_id).strip()
    b_id_col = 'Pseudonym' if 'Pseudonym' in blood_raw.columns else ('Patient_ID' if 'Patient_ID' in blood_raw.columns else 'ID')
    m_id_col = 'Pseudonym' if 'Pseudonym' in bm_raw.columns else ('Patient_ID' if 'Patient_ID' in bm_raw.columns else 'ID')

    blood_df = blood_raw[blood_raw[b_id_col].astype(str).str.strip() == p_id].copy()
    bm_df = bm_raw[bm_raw[m_id_col].astype(str).str.strip() == p_id].copy()

    # Exact preprocessing match 
    if not blood_df.empty:
        blood_df.loc[blood_df['days_lab'] < 0, 'days_lab'] = 0
        blood_df = blood_df.groupby('days_lab', as_index=False)['b_neut'].mean().sort_values('days_lab')
        blood_df.loc[blood_df['b_neut'] <= 0, 'b_neut'] = 1e-9
        blood_df['b_neut_scaled'] = blood_df['b_neut'] * 5 * 10**9 / 80

    if not bm_df.empty:
        bm_df.loc[bm_df['days_from_bm'] < 0, 'days_from_bm'] = 0
        bm_df = bm_df.groupby('days_from_bm', as_index=False)['bm_blasts'].mean().sort_values('days_from_bm')

    schedule = extract_schedule(target_patient_id, drug_raw)

    p_row = merged_df[merged_df['Study_subject'].astype(str).str.strip() == p_id].iloc[0]
    p = base_params.copy()
    phsc = float(p_row['Predefined_pHSC'])
    coef = float(p_row['Predefined_Coef'])
    p['p1l'] = p['p1'] * coef
    p['p_HSCbyLSC'] = phsc
    p['p_LSCbyHSC'] = 1.0 - phsc
    p['ven_kill_rate_Blast'] = p['ven_kill_rate_LSC'] = float(p_row['Fitted_DrugKill_LSC'])
    p['ven_kill_rate_HSC'] = p['ven_kill_rate_Neut'] = float(p_row['Fitted_DrugKill_HSC'])

    initial_neut = blood_df['b_neut_scaled'].iloc[0] if not blood_df.empty else p['healthy_neut_per_kg']
    initial_blast_perc = (bm_df['bm_blasts'].iloc[0] / 100.0) if not bm_df.empty else 0.50
    z0 = get_diagnosis_state(p, initial_blast_perc, initial_neut)
    if z0 is None:
        z0 = [p['niche_capacity'] * (1.0 - initial_blast_perc), initial_neut, p['niche_capacity'] * initial_blast_perc, 0.0]

    min_t = min(blood_df['days_lab'].min() if not blood_df.empty else 0, bm_df['days_from_bm'].min() if not bm_df.empty else 0, 0)
    max_t = max(blood_df['days_lab'].max() if not blood_df.empty else 0, bm_df['days_from_bm'].max() if not bm_df.empty else 0) + 10
    t_sim, y_sim = run_phased_simulation(min_t, max_t, z0, p, schedule)
    sim_hsc, sim_neut, sim_lsc, sim_blast = y_sim
    sim_blast_perc = 100.0 * (sim_lsc + sim_blast) / (sim_hsc + sim_lsc + sim_blast + 1e-12)

    # --- PANELS B & C: SURVIVAL ANALYSIS ---
    df = merged_df.copy()
    df['OS_time'] = pd.to_numeric(df['OS_time'], errors='coerce')
    df['OS_status'] = pd.to_numeric(df['OS_status'], errors='coerce').fillna(0).astype(int)

    # 1. Clip raw parameters to prevent runaway zeroes or extreme outliers
    df['Fitted_DrugKill_LSC_clean'] = df['Fitted_DrugKill_LSC'].clip(lower=1e-6, upper=5.0)
    df['Fitted_DrugKill_HSC_clean'] = df['Fitted_DrugKill_HSC'].clip(lower=1e-6, upper=5.0)

    # 2. Standardize covariates
    df['pHSC_z'] = (df['Predefined_pHSC'] - df['Predefined_pHSC'].mean()) / (df['Predefined_pHSC'].std() or 1.0)
    df['coef_z'] = (df['Predefined_Coef'] - df['Predefined_Coef'].mean()) / (df['Predefined_Coef'].std() or 1.0)
    df['Fitted_DrugKill_LSC_z'] = (df['Fitted_DrugKill_LSC_clean'] - df['Fitted_DrugKill_LSC_clean'].mean()) / (df['Fitted_DrugKill_LSC_clean'].std() or 1.0)
    df['Fitted_DrugKill_HSC_z'] = (df['Fitted_DrugKill_HSC_clean'] - df['Fitted_DrugKill_HSC_clean'].mean()) / (df['Fitted_DrugKill_HSC_clean'].std() or 1.0)

    # 3. Construct df_clean
    surv_cols = ['OS_time', 'OS_status', 'pHSC_z', 'coef_z', 'Fitted_DrugKill_LSC_z', 'Fitted_DrugKill_HSC_z']
    df_clean = df[surv_cols].dropna().reset_index(drop=True)

    # Model B: Static Baseline
    cph_static = CoxPHFitter(penalizer=0.01)
    cph_static.fit(df_clean, duration_col='OS_time', event_col='OS_status', formula="coef_z + pHSC_z")
    score_static = np.asarray(cph_static.predict_log_partial_hazard(df_clean)).ravel()
    cut_static, p_val_static = find_optimal_cutpoint(df_clean.assign(score=score_static), 'score', 'OS_time', 'OS_status')
    group_static = np.where(score_static <= cut_static, 'Low Risk Profile', 'High Risk Profile')

    # Model C: Extended Time-Series (Baseline + Dynamic Kill Interaction)
    cph_ext = CoxPHFitter(penalizer=0.01)
    cph_ext.fit(
        df_clean,
        duration_col='OS_time',
        event_col='OS_status',
        formula="coef_z + pHSC_z + Fitted_DrugKill_LSC_z * Fitted_DrugKill_HSC_z"
    )
    score_ext = np.asarray(cph_ext.predict_log_partial_hazard(df_clean)).ravel()
    cut_ext, p_val_ext = find_optimal_cutpoint(df_clean.assign(score=score_ext), 'score', 'OS_time', 'OS_status')
    group_ext = np.where(score_ext <= cut_ext, 'Low Risk Profile', 'High Risk Profile')

    # --- CANVAS ASSEMBLY (Larger figure, tighter spacing, zero dead gaps) ---
    fig = plt.figure(figsize=(24, 7.8), dpi=300)
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.0, 1.0], hspace=0.03, wspace=0.18)
    ax_a1 = fig.add_subplot(gs[0, 0])
    ax_a2 = fig.add_subplot(gs[1, 0], sharex=ax_a1)
    ax_b, ax_c = fig.add_subplot(gs[:, 1]), fig.add_subplot(gs[:, 2])

    # A1: Neutrophils
    ax_a1.text(-0.06, 1.08, 'A', transform=ax_a1.transAxes, fontsize=28, fontweight='bold', va='top', ha='right')
    ax_a1.plot(t_sim, sim_neut, color=c_blue, linewidth=2.6, label="Simulated Neutrophils")
    if not blood_df.empty:
        ax_a1.scatter(blood_df['days_lab'], blood_df['b_neut_scaled'], color="black", s=45, zorder=3, label="Clinical Data")
    ax_a1.set_ylabel("Neutrophil\nCounts", fontsize=15, fontweight="bold", labelpad=8)
    ax_a1.tick_params(axis='both', labelsize=13)
    ax_a1.yaxis.offsetText.set_fontsize(12)
    plt.setp(ax_a1.get_xticklabels(), visible=False)
    ax_a1.legend(loc="upper right", frameon=True, fontsize=12)

    # A2: Blasts
    ax_a2.plot(t_sim, sim_blast_perc, color=c_orange, linewidth=2.6, label="Simulated Blasts %")
    if not bm_df.empty:
        ax_a2.scatter(bm_df['days_from_bm'], bm_df['bm_blasts'], color="black", marker="s", s=45, zorder=3, label="Clinical Data")
    ax_a2.set_ylabel("Bone Marrow\nBlasts (%)", fontsize=15, fontweight="bold", labelpad=8)
    ax_a2.set_xlabel("Time (Days)", fontsize=16, fontweight="bold", labelpad=8)
    ax_a2.set_ylim(-4, 105)
    ax_a2.tick_params(axis='both', labelsize=14)

    legend_added = False
    for start, stop in schedule:
        ax_a1.axvspan(start, stop, color=c_drug, alpha=0.6, zorder=1)
        ax_a2.axvspan(start, stop, color=c_drug, alpha=0.6, label="Drug Active" if not legend_added else "", zorder=1)
        legend_added = True
    ax_a2.legend(loc="upper right", frameon=True, fontsize=12)

    # B: Static Baseline KM
    ax_b.text(-0.04, 1.04, 'B', transform=ax_b.transAxes, fontsize=28, fontweight='bold', va='top', ha='right')
    kmf = KaplanMeierFitter()
    for grp, col, ls in [('Low Risk Profile', c_blue, '-'), ('High Risk Profile', c_orange, '--')]:
        mask = group_static == grp
        if mask.sum() > 0:
            kmf.fit(df_clean.loc[mask, 'OS_time'], df_clean.loc[mask, 'OS_status'], label=f"{grp} (N={mask.sum()})")
            kmf.plot_survival_function(ax=ax_b, color=col, linewidth=3.5, linestyle=ls, ci_show=False)
    ax_b.set_xlabel('Time (Days)', fontsize=16, fontweight='bold', labelpad=8)
    ax_b.set_ylabel('Overall Survival Probability', fontsize=16, fontweight='bold', labelpad=8)
    ax_b.set_ylim(-0.02, 1.02)
    ax_b.tick_params(axis='both', labelsize=14)
    p_str_b = f"$p = {p_val_static:.4e}$" if p_val_static < 0.001 else f"$p = {p_val_static:.4f}$"
    ax_b.text(0.42, 0.18, f"Static Baseline Model\nC-Index = {cph_static.concordance_index_:.3f}\nLog-Rank {p_str_b.replace('e-0', 'e-')}",
              transform=ax_b.transAxes, fontsize=13, fontweight='bold', bbox=dict(facecolor='white', alpha=0.92, edgecolor='#cccccc', boxstyle='round,pad=0.5'))
    ax_b.legend(loc='upper right', frameon=False, fontsize=13)

    # C: Extended Time-Series KM
    ax_c.text(-0.04, 1.04, 'C', transform=ax_c.transAxes, fontsize=28, fontweight='bold', va='top', ha='right')
    for grp, col, ls in [('Low Risk Profile', c_blue, '-'), ('High Risk Profile', c_orange, '--')]:
        mask = group_ext == grp
        if mask.sum() > 0:
            kmf.fit(df_clean.loc[mask, 'OS_time'], df_clean.loc[mask, 'OS_status'], label=f"{grp} (N={mask.sum()})")
            kmf.plot_survival_function(ax=ax_c, color=col, linewidth=3.5, linestyle=ls, ci_show=False)
    ax_c.set_xlabel('Time (Days)', fontsize=16, fontweight='bold', labelpad=8)
    ax_c.set_ylabel('Overall Survival Probability', fontsize=16, fontweight='bold', labelpad=8)
    ax_c.set_ylim(-0.02, 1.02)
    ax_c.tick_params(axis='both', labelsize=14)
    p_str_c = f"$p = {p_val_ext:.4e}$" if p_val_ext < 0.001 else f"$p = {p_val_ext:.4f}$"
    ax_c.text(0.42, 0.18, f"Extended Time-Series Model\nC-Index = {cph_ext.concordance_index_:.3f}\nLog-Rank {p_str_c.replace('e-0', 'e-')}",
              transform=ax_c.transAxes, fontsize=13, fontweight='bold', bbox=dict(facecolor='white', alpha=0.92, edgecolor='#cccccc', boxstyle='round,pad=0.5'))
    ax_c.legend(loc='upper right', frameon=False, fontsize=13)

    for ax in [ax_a1, ax_a2, ax_b, ax_c]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.subplots_adjust(left=0.05, right=0.99, top=0.94, bottom=0.10)
    out_file = Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    png_path = out_file.with_suffix(".png")
    pdf_path = out_file.with_suffix(".pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight", pad_inches=0.08)
    plt.savefig(pdf_path, dpi=300, bbox_inches="tight", pad_inches=0.08)
    print(f"Figures successfully saved to: {png_path} and {pdf_path}")
    plt.show()

# =====================================================================
# 3. RUN PLOTTING PIPELINE
# =====================================================================
if __name__ == "__main__":
    atlas_df = pd.read_csv("IntermediateResults/Patient_Atlas.csv")
    traj_path = "InputData/treatment_trajectories.xlsx"
    sheet_dict = pd.read_excel(traj_path, sheet_name=None)
    blood_all = sheet_dict.get('Neutrophil').copy()
    bm_all = sheet_dict.get('BM_blast').copy()
    drug_all = sheet_dict.get('Cycles').copy()

    # Scale neutrophils from 10^9/L to cells/kg and align pre-treatment days
    b_id_col = 'Pseudonym' if 'Pseudonym' in blood_all.columns else ('Patient_ID' if 'Patient_ID' in blood_all.columns else 'ID')
    blood_all.loc[blood_all['days_lab'] < 0, 'days_lab'] = 0
    blood_all = blood_all.groupby([b_id_col, 'days_lab'], as_index=False)['b_neut'].mean()
    blood_all.loc[blood_all['b_neut'] <= 0, 'b_neut'] = 1e-9
    blood_all['b_neut'] = blood_all['b_neut'] * 5 * 10**9 / 80

    # Align pre-treatment bone marrow day
    m_id_col = 'Pseudonym' if 'Pseudonym' in bm_all.columns else ('Patient_ID' if 'Patient_ID' in bm_all.columns else 'ID')
    bm_all.loc[bm_all['days_from_bm'] < 0, 'days_from_bm'] = 0
    bm_all = bm_all.groupby([m_id_col, 'days_from_bm'], as_index=False)['bm_blasts'].mean()

    # 1. Generate individual patient fit plots
    generate_patient_fit_plots(atlas_df, blood_all, bm_all, drug_all, base_params, output_dir="Plots")

    # 2. Merge with clinical data and generate Figure 4
    clin_labs = pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")
    clin_labs['Study_subject'] = clin_labs['Study_subject'].astype(str).str.strip()
    atlas_df['Patient_ID'] = atlas_df['Patient_ID'].astype(str).str.strip()
    merged_final = pd.merge(clin_labs, atlas_df, left_on="Study_subject", right_on="Patient_ID", how="inner")

    generate_figure_three_panels(
        merged_df=merged_final,
        target_patient_id=9851,
        excel_path=traj_path,
        output_path="Figures/Figure4.png"
    )